<a href="https://colab.research.google.com/github/xesmaze/Causal-Deconvolution-of-Networks/blob/master/MB_for_GraphPruning_in_large_Biological_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Rather than using MB primarily to identify “the causal predictors of yield,” you can use it as a **graph-pruning rule**: keep only the local conditional-dependence structure needed to preserve information about each node, and eliminate edges that become redundant after conditioning on the appropriate blanket.

The logic would be:

$$
X_i \perp X_{\setminus(\{i\}\cup MB_i)} \mid MB(X_i)
$$

So once the Markov Blanket of a node \(X_i\) is known, variables outside that blanket should not need direct connections to \(X_i\). In a dense metabolite/enzyme/phenotype network, that gives you a principled way to remove a huge number of biologically tempting but statistically redundant edges.

For example, suppose the original association graph looks like:

$$
G \rightarrow M_1,M_2,M_3,M_4
$$

$$
M_1\leftrightarrow M_2\leftrightarrow M_3\leftrightarrow M_4
$$

with essentially all metabolites also connected to enzymes, leaf N, sugars, and yield. Pairwise correlation will produce a “hairball.”

But imagine conditional-independence testing gives:

$$
M_1 \rightarrow E_1 \rightarrow LeafN \rightarrow GY
$$

and

$$
M_2 \rightarrow E_1
$$

while

$$
GY\perp M_1,M_2,E_1\mid LeafN.
$$

Then for the local graph around yield,

$$
MB(GY)=\{LeafN,\ldots\}
$$

and you can prune direct edges such as

$$
M_1\rightarrow GY,\qquad
M_2\rightarrow GY,\qquad
E_1\rightarrow GY.
$$

They may all correlate with yield, but they do **not require direct edges**.

That distinction is exactly what would make this useful.

### Build the analysis in two stages

Start with an intentionally permissive biological graph. Edges could initially come from significant associations, known biochemical relationships, pathway membership, or a causal-discovery algorithm.

So you might begin with something dense:

$$
G \rightarrow
\{\text{metabolites, enzymes, physiology}\}
$$

$$
M \leftrightarrow M
$$

$$
M\leftrightarrow E
$$

$$
E\leftrightarrow P
$$

$$
M,E,P\rightarrow Y.
$$

Then use Markov Blankets to ask for every node:

> Which neighbors are actually required to render this node conditionally independent of the rest of the system?

Edges not supported by that local structure become candidates for pruning.

Conceptually:

$$
\text{Dense biological network}
\rightarrow
\text{conditional-independence testing}
\rightarrow
\text{MB neighborhoods}
\rightarrow
\text{pruned graph}.
$$

### Do not literally take the union of every MB, however

There is an important subtlety.

If you simply calculate

$$
MB(X_1),MB(X_2),\ldots,MB(X_p)
$$

and retain every pair appearing together, you don't automatically obtain a causal DAG. A Markov Blanket contains:

$$
Parents(X)\cup Children(X)\cup Spouses(X),
$$

and spouses do **not necessarily have a direct edge**.

For example:

$$
A\rightarrow C\leftarrow B.
$$

The Markov Blanket of $A$ contains $B$, because $B$ is the other parent of $C$:

$$
B\in MB(A).
$$

But there is no edge

$$
A-B.
$$

So I would use MB for **candidate-edge reduction**, followed by actual conditional-independence/orientation tests.

That yields something like:

$$
\text{MB pruning}
\rightarrow
\text{skeleton learning}
\rightarrow
\text{edge orientation}.
$$

### That approach fits this maize dataset especially well

You have multiple biological layers:

$$
G
\rightarrow
M
\rightarrow
E
\rightarrow
P
\rightarrow
Y
$$

but in reality all of those layers have enormous within-layer correlation.

For instance, dozens of amino acids may correlate with one another because they share nitrogen-assimilation biology. You don't want:

$$
Ala\rightarrow Y
$$

$$
Glu\rightarrow Y
$$

$$
Gln\rightarrow Y
$$

$$
Asp\rightarrow Y
$$

simply because all four correlate with grain yield.

MB pruning might instead produce something like:

$$
Gln
\rightarrow
GS
\rightarrow
LeafN
\rightarrow
GY
$$

with the other amino acids connected locally to the nitrogen-metabolism network but **not directly to yield**.

That graph is biologically much more interpretable.

### And preserve temporal constraints while pruning

This is where your dataset gives you a major advantage.

You can impose:

$$
G \prec V \prec 15DAS \prec Y.
$$

So if the statistical graph suggests

$$
Y\rightarrow M_{15}
$$

you simply forbid that orientation.

Likewise:

$$
M_{15}\rightarrow M_V
$$

is prohibited.

Therefore MB determines **which connections survive**, while biological time determines **which directions are permissible**.

I would think about it as:

$$
\boxed{\text{MB = sparsity constraint}}
$$

rather than

$$
\boxed{\text{MB = causal inference}}
$$

and then use temporal/biochemical knowledge plus collider/orientation rules to construct the causal graph.

### An especially nice implementation would be iterative pruning

Start with $G_0$, your dense candidate graph.

For each node $X_i$, estimate:

$$
MB_i=MB(X_i).
$$

Then remove candidate adjacency $X_i-X_j$ when neither local neighborhood supports it, subject to your causal-discovery criterion.

So:

$$
G_0
\rightarrow G_1
\rightarrow G_2
\rightarrow\cdots\rightarrow G^*
$$

until the edge set stabilizes.

And bootstrap the process.

You can assign every retained edge a stability score:

$$
S_{ij}
=
\frac{1}{B}
\sum_{b=1}^B
I[(X_i,X_j)\in E_b].
$$

Then perhaps only display edges with

$$
S_{ij}>0.7
$$

or whatever threshold proves appropriate.

That would give you a **stable sparse biological graph**, rather than a single fragile inferred network.

The conceptual goal, then, is not *“find the Markov Blanket of yield.”* It is **“use local Markov Blankets across the system to prune a high-dimensional biological association network into a sparse causal candidate graph.”** I think that framing is substantially stronger.